**This notebook is essentially a rough work. I have created processing.py to clean and process data, please use the functionalities of that**

In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Data is huge, ~15M records per month. Use only necessary columns.
# Load last three month data and make ready a pandas dataframe.

def load_month(path):
    return pd.read_parquet(path, columns=REQ_COLUMNS)

REQ_COLUMNS = [
    'station_name',
    'xml_station_name',
    'final_destination_station',
    'train_name',
    'delay_in_min',
    'time',
    'is_canceled',
    'train_type'
]

data_dir = Path('data')

files = [
    data_dir / "data-2025-10.parquet",
    data_dir / "data-2025-11.parquet",
    data_dir / "data-2025-12.parquet",
]

df = pd.concat([load_month(f) for f in files], ignore_index=True)

In [ ]:
# Basic cleaning

# Only focus on RE and ICE
df = df[df['train_type'] != 'S']

# If station name is null, replace it with xml_station_name
df["station_name"] = df["station_name"].fillna(df["xml_station_name"])

# Remove cancelled trains, to focus delay magnitude
df = df[
    (df["is_canceled"] == False) &
    (df["delay_in_min"].notna())
]

# Have proper formatting
df["time"] = pd.to_datetime(df["time"])
df["date"] = df["time"].dt.date

# These columns are no longer required
df.drop(['is_canceled', 'xml_station_name'], axis=1, inplace=True)

In [ ]:
# Create a new column to uniquely identify the route

df["route_id"] = (
    df["train_name"].astype(str) +  '?'  + df["station_name"].astype(str)
)

In [ ]:
# Contains ~2M records. Is not pushed to git, please run this to save locally.
# This Data is needed
df.to_csv("data/db_delays_cleaned.csv")

In [ ]:
# Sample data: only include top and busiest routes

route_counts = (
    df.groupby("route_id")
      .size()
      .sort_values(ascending=False)
)

# Keep top N routes
TOP_N_ROUTES = 150

major_routes = route_counts.head(TOP_N_ROUTES).index

df = df[df["route_id"].isin(major_routes)]

In [ ]:
daily_route_delays = (
    df.groupby(["route_id", "date"])
      .agg(
          mean_delay=("delay_in_min", "mean"),
          sd_delay=("delay_in_min", "std"),
          n_trains=("delay_in_min", "size")
      )
      .reset_index()
)


daily_route_delays["date"] = pd.to_datetime(daily_route_delays["date"])
daily_route_delays["weekday"] = daily_route_delays["date"].dt.weekday
daily_route_delays["is_weekend"] = daily_route_delays["weekday"] >= 5

In [ ]:
daily_route_delays.to_csv("data/daily_route_delays.csv")

In [ ]:
daily_route_delays['n_trains']

### Read stations data and try to see if its a match for our original data. 

Data: https://www.kaggle.com/datasets/mexwell/deutsche-bahn-train-stops-germany

In [ ]:

db = pd.read_excel("data/db_data.xlsx")
db = db[['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3']]

In [ ]:
def concat(t):
    return "".join([str(i).replace('"', '') for i in t[1]]).split(';')

In [ ]:
rows = [concat(i) for i in db.iterrows()]
rows = pd.DataFrame(rows).drop([27, 28], axis=1)

In [ ]:
cols = "EVA_NR;DS100;IFOPT;db_name;traffic;operator_name;operator_nr;status;distance;formatted;lat;lon;housenumber;name;street;postcode;district;suburb;county;state;state_code;city;country;country_code;attribution;attribution_license;attribution_url"
cols = cols.split(";")
rows.columns = cols

In [ ]:
df[df['eva']=='8001944']

In [ ]:
rows

In [ ]:
# Another source of station data
d = pd.read_csv('data/worldcities.csv')
d[d['country'] == 'Germany']

In [ ]:
st = pd.read_csv("data/stations_dataset.csv")

In [ ]:
df